# The first fine-tune

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/qwen-symbolic-thinking/blob/main/notebooks/train.ipynb)

A LoRA adapter on Qwen3.5-9B, taught the chains of
[thought-symbolic-image-captions](https://huggingface.co/datasets/AbstractPhil/thought-symbolic-image-captions) with a
one-line prompt: about 980 picture rows (COCO 2017 images fetched by id) and as many specification rows. The adapter ships
to [qwen-symbolic-thinking-image-training](https://huggingface.co/AbstractPhil/qwen-symbolic-thinking-image-training)
under `lora/<model>/r<rank>/<stamp>/` during training; afterwards the trained model is read on the screen set in the
one-line forms, beside the base model's screen.

Runtime: a GPU (the RTX PRO 6000 is the reference card). Secrets: `HF_TOKEN` with notebook access.

Run C1, then **Runtime > Restart session**, then C2 and C3 in turn (not Run all).

In [ ]:
# C1: install, then Runtime > Restart session
import importlib.util, subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("git+https://github.com/AbstractEyes/qwen-symbolic-thinking@v0.1.0", "transformers>=5.0", "accelerate>=1.0", "peft>=0.18")

FAST_PATH = True   # the linear-attention kernels Qwen3.5 uses; causal-conv1d compiles for about 5 minutes; all or nothing
if FAST_PATH:
    if all(importlib.util.find_spec(m) for m in ("causal_conv1d", "fla", "triton")):
        print("fast path already present")
    else:
        try:
            pip("causal-conv1d", "--no-build-isolation")
            pip("flash-linear-attention", "triton")
            print("fast path on")
        except subprocess.CalledProcessError as e:
            print("fast path off (training still runs, slower):", e)
            subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "causal-conv1d"])
print("installed; now Runtime > Restart session, then run C2")

In [ ]:
# C2: train (progress lines show the time left; the adapter ships every 100 steps and at the end)
from qwen_symbolic_thinking.train import run as train

trained = train(model_id="Qwen/Qwen3.5-9B", inputs="hub:sft/sft_v1.json", epochs=1,
                rank=32, alpha=64, lr=2e-4, batch=4, grad_accum=4, max_len=2048)

In [ ]:
# C3: the trained model on the screen set, one-line prompt forms
from qwen_symbolic_thinking.screen import run as screen

summary = screen(models=["Qwen/Qwen3.5-9B"], inputs="hub:screens/inputs/screen_v1.json",
                 forms=("short", "spec_short"), adapter=trained["adapter_dir"], tag="lora_r32",
                 batch=4, max_new_tokens=700)